In [5]:
import pandas as pd
import numpy as np
import pymannkendall as mk
import matplotlib.pyplot as plt

def load_and_analyze(file_path, sheet_name, dataset_name):
    data = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    
    
    print(f"Data shape for {dataset_name}: {data.shape}")
    
   
    years = data.iloc[0, 4:23].values  
    stations = data.iloc[1:, 1].values  
    
    if data.shape[1] < 25:
        raise ValueError(f"The Excel file for {dataset_name} does not have enough columns. Expected at least 25 columns.")
    
    well_numbers = data.iloc[1:26, 23].values  
    district_names = data.iloc[1:26, 24].values  
    
    if len(stations) != len(well_numbers) or len(stations) != len(district_names):
        raise ValueError(f"Mismatch in the number of stations, well numbers, or district names in {dataset_name}.")
    
    groundwater_levels = data.iloc[1:, 4:23].values

    results = []

    for i, station in enumerate(stations):
        levels = groundwater_levels[i]
        levels = pd.to_numeric(levels, errors='coerce')
        valid_indices = ~np.isnan(levels)
        
        valid_years = pd.to_numeric(years, errors='coerce')
        if np.all(np.isnan(valid_years)):
            print(f"Skipping Station {station} in {dataset_name}: All years are NaN.")
            continue
        
        valid_years = valid_years[valid_indices]
        valid_levels = levels[valid_indices]
        
        if len(valid_levels) < 2:
            print(f"Skipping Station {station} in {dataset_name}: Insufficient data points for analysis.")
            continue
        
        try:
            mk_result = mk.original_test(valid_levels, alpha=0.05)
            sen_slope = mk.sens_slope(valid_levels)
        except Exception as e:
            print(f"Error in analysis for Station {station} in {dataset_name}: {e}")
            continue
        
        results.append({
            'Station': station,
            'Well Number': well_numbers[i],
            'District Name': district_names[i],
            'Dataset': dataset_name,
            'MK Z Value': mk_result.z,
            'MK Trend': mk_result.trend,
            'MK Tau': mk_result.Tau,
            'Sen\'s Slope': sen_slope.slope,
            'Intercept': sen_slope.intercept,
            'Valid Years': valid_years,
            'Valid Levels': valid_levels
        })
    
    return results


# Datasets
pre_monsoon_file = './spre.xlsx'
post_monsoon_file = './spost.xlsx'
sheet_name = 'sheet_1'

pre_monsoon_results = load_and_analyze(pre_monsoon_file, sheet_name, 'Pre-Monsoon')
post_monsoon_results = load_and_analyze(post_monsoon_file, sheet_name, 'Post-Monsoon')


combined_results = {}

for result in pre_monsoon_results:
    combined_results[result['Station']] = {'Pre-Monsoon': result}

for result in post_monsoon_results:
    if result['Station'] in combined_results:
        combined_results[result['Station']]['Post-Monsoon'] = result
    else:
        combined_results[result['Station']] = {'Post-Monsoon': result}


for station, data in combined_results.items():
    plt.figure(figsize=(12, 7))
    
   
    if 'Pre-Monsoon' in data:
        well_number = data['Pre-Monsoon']['Well Number']
        district_name = data['Pre-Monsoon']['District Name']
    else:
        well_number = data['Post-Monsoon']['Well Number']
        district_name = data['Post-Monsoon']['District Name']
    
    all_levels = []
    
    # Plotting Pre-Monsoon data
    if 'Pre-Monsoon' in data:
        pre = data['Pre-Monsoon']
        years = pre['Valid Years']
        levels = pre['Valid Levels']
        slope = pre["Sen's Slope"]
        intercept = pre["Intercept"]
        
        plt.plot(years, levels, 'b-o', label='Pre-Monsoon Data')
        plt.plot(years, slope * years + intercept, 
                color='navy', linestyle='--', 
                label=f'Pre Trend ({slope:.2f} m/yr)')
        all_levels.extend(levels)
    
    # Plotting Post-Monsoon data
    if 'Post-Monsoon' in data and len(data['Post-Monsoon']['Valid Years']) > 0:
        post = data['Post-Monsoon']
        years = post['Valid Years']
        levels = post['Valid Levels']
        slope = post["Sen's Slope"]
        intercept = post["Intercept"]
        
        plt.plot(years, levels, 'g-s', label='Post-Monsoon Data')
        plt.plot(years, slope * years + intercept, 
                color='darkgreen', linestyle='--', 
                label=f'Post Trend ({slope:.2f} m/yr)')
        all_levels.extend(levels)
    
    
    plt.title(f"Groundwater Trend: Well {well_number}\n{station}, {district_name}")
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Groundwater Depth (m)', fontsize=12)
    plt.gca().invert_yaxis()
    
    
    if all_levels:
        y_padding = (max(all_levels) - min(all_levels)) * 0.1
        plt.ylim(max(all_levels) + y_padding, max(0, min(all_levels) - y_padding))
    else:
        plt.ylim(10, 0)
    
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.xlim(1995, 2025)
    
   
    filename = f"GW_Trend_Well_{well_number}_{district_name.replace(' ', '_')}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

print("All plots generated successfully!")

Data shape for Pre-Monsoon: (26, 25)
Data shape for Post-Monsoon: (26, 25)
All plots generated successfully!
